In [1]:
# Cell 1 — Import libraries and set display options

import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

print("Libraries imported.")


Libraries imported.


In [2]:
# Cell 2 — Load preprocessed datasets with cluster labels

processed_path = "../data/processed"

train_df = pd.read_csv(f"{processed_path}/train_with_clusters.csv")
val_df   = pd.read_csv(f"{processed_path}/val_with_clusters.csv")
test_df  = pd.read_csv(f"{processed_path}/test_with_clusters.csv")

print("Train shape:", train_df.shape)
print("Val shape:  ", val_df.shape)
print("Test shape: ", test_df.shape)

train_df.head()


Train shape: (7868, 13)
Val shape:   (1681, 13)
Test shape:  (1686, 13)


,Age,Gender,Occupation,Country,Sleep_Hours_Scaled,Work_Hours_Scaled,Physical_Activity_Hours_Scaled,Social_Media_Usage_Scaled,Diet_Quality_Encoded_Scaled,Smoking_Habit_Encoded_Scaled,Alcohol_Consumption_Encoded_Scaled,Stress_Level,Cluster
0,21,Non-binary,Finance,Other,0.133333,0.14,0.2,0.527273,0.0,1.000000,0.000000,Medium,0
1,24,Non-binary,Finance,USA,0.916667,0.68,0.5,0.563636,0.0,0.000000,0.333333,Low,0
2,52,Female,Sales,Canada,0.183333,0.60,0.3,0.090909,1.0,0.666667,1.000000,High,2
3,35,Female,Engineering,Other,0.633333,0.92,0.8,0.145455,0.5,0.000000,0.000000,Low,0
4,42,Non-binary,Finance,UK,0.416667,0.96,0.8,0.781818,0.5,0.333333,1.000000,High,1


In [3]:
# Cell 3 — Define lifestyle features, cluster feature, and target

target_col = "Stress_Level"

# 7 lifestyle habits (scaled)
lifestyle_features = [
    col for col in train_df.columns
    if col.endswith("_Scaled") and not col.startswith("Age")
]

print("Lifestyle features:")
print(lifestyle_features)

# Cluster column created in Notebook 03
cluster_feature = ["Cluster"]

# Model A = lifestyle only (baseline)
features_A = lifestyle_features.copy()

# Model B = lifestyle + cluster (main model in thesis)
features_B = lifestyle_features + cluster_feature

print("\nModel A features (Lifestyle only):")
print(features_A)

print("\nModel B features (Lifestyle + Cluster):")
print(features_B)

print("\nUnique Stress Levels in training set:")
print(train_df[target_col].unique())


Lifestyle features:
['Sleep_Hours_Scaled', 'Work_Hours_Scaled', 'Physical_Activity_Hours_Scaled', 'Social_Media_Usage_Scaled', 'Diet_Quality_Encoded_Scaled', 'Smoking_Habit_Encoded_Scaled', 'Alcohol_Consumption_Encoded_Scaled']

Model A features (Lifestyle only):
['Sleep_Hours_Scaled', 'Work_Hours_Scaled', 'Physical_Activity_Hours_Scaled', 'Social_Media_Usage_Scaled', 'Diet_Quality_Encoded_Scaled', 'Smoking_Habit_Encoded_Scaled', 'Alcohol_Consumption_Encoded_Scaled']

Model B features (Lifestyle + Cluster):
['Sleep_Hours_Scaled', 'Work_Hours_Scaled', 'Physical_Activity_Hours_Scaled', 'Social_Media_Usage_Scaled', 'Diet_Quality_Encoded_Scaled', 'Smoking_Habit_Encoded_Scaled', 'Alcohol_Consumption_Encoded_Scaled', 'Cluster']

Unique Stress Levels in training set:
['Medium' 'Low' 'High']


In [4]:
# Cell 4 — Prepare X and y for both models (same y, different X)

# Targets (categorical: "Low", "Medium", "High")
y_train = train_df[target_col]
y_val   = val_df[target_col]
y_test  = test_df[target_col]

# MODEL A — Lifestyle only
X_train_A = train_df[features_A]
X_val_A   = val_df[features_A]
X_test_A  = test_df[features_A]

# MODEL B — Lifestyle + Cluster
X_train_B = train_df[features_B]
X_val_B   = val_df[features_B]
X_test_B  = test_df[features_B]

print("Model A shapes:")
print("  X_train_A:", X_train_A.shape, " y_train:", y_train.shape)
print("  X_val_A:  ", X_val_A.shape,   " y_val:", y_val.shape)
print("  X_test_A: ", X_test_A.shape,  " y_test:", y_test.shape)

print("\nModel B shapes:")
print("  X_train_B:", X_train_B.shape)
print("  X_val_B:  ", X_val_B.shape)
print("  X_test_B: ", X_test_B.shape)


Model A shapes:
  X_train_A: (7868, 7)  y_train: (7868,)
  X_val_A:   (1681, 7)  y_val: (1681,)
  X_test_A:  (1686, 7)  y_test: (1686,)

Model B shapes:
  X_train_B: (7868, 8)
  X_val_B:   (1681, 8)
  X_test_B:  (1686, 8)


In [5]:
# Cell 5 — Train Model A (Multinomial Logistic Regression, lifestyle only)

model_A = LogisticRegression(
    multi_class="multinomial",
    solver="lbfgs",
    max_iter=1000,
    random_state=42
)

model_A.fit(X_train_A, y_train)

print("Model A trained (lifestyle only).")


Model A trained (lifestyle only).


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [6]:
# Cell 6 — Evaluate Model A on validation and test sets

# Validation
val_pred_A = model_A.predict(X_val_A)
val_acc_A = accuracy_score(y_val, val_pred_A)

print("===== MODEL A (Lifestyle Only) — VALIDATION =====")
print("Validation Accuracy:", round(val_acc_A, 4))
print(classification_report(y_val, val_pred_A, digits=3))

# Test
test_pred_A = model_A.predict(X_test_A)
test_acc_A = accuracy_score(y_test, test_pred_A)

print("\n===== MODEL A (Lifestyle Only) — TEST =====")
print("Test Accuracy:", round(test_acc_A, 4))
print(classification_report(y_test, test_pred_A, digits=3))


===== MODEL A (Lifestyle Only) — VALIDATION =====
Validation Accuracy: 0.934
              precision    recall  f1-score   support

        High      0.958     0.958     0.958       806
         Low      0.950     0.953     0.952       443
      Medium      0.872     0.870     0.871       432

    accuracy                          0.934      1681
   macro avg      0.927     0.927     0.927      1681
weighted avg      0.934     0.934     0.934      1681


===== MODEL A (Lifestyle Only) — TEST =====
Test Accuracy: 0.9371
              precision    recall  f1-score   support

        High      0.960     0.953     0.957       809
         Low      0.977     0.941     0.959       444
      Medium      0.859     0.903     0.881       433

    accuracy                          0.937      1686
   macro avg      0.932     0.932     0.932      1686
weighted avg      0.939     0.937     0.938      1686



In [7]:
# Cell 7 — Train Model B (Multinomial Logistic Regression, lifestyle + cluster)

model_B = LogisticRegression(
    multi_class="multinomial",
    solver="lbfgs",
    max_iter=1000,
    random_state=42
)

model_B.fit(X_train_B, y_train)

print("Model B trained (lifestyle + cluster).")


Model B trained (lifestyle + cluster).


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [8]:
# Cell 8 — Evaluate Model B on validation and test sets

# Validation
val_pred_B = model_B.predict(X_val_B)
val_acc_B = accuracy_score(y_val, val_pred_B)

print("===== MODEL B (Lifestyle + Cluster) — VALIDATION =====")
print("Validation Accuracy:", round(val_acc_B, 4))
print(classification_report(y_val, val_pred_B, digits=3))

# Test
test_pred_B = model_B.predict(X_test_B)
test_acc_B = accuracy_score(y_test, test_pred_B)

print("\n===== MODEL B (Lifestyle + Cluster) — TEST =====")
print("Test Accuracy:", round(test_acc_B, 4))
print(classification_report(y_test, test_pred_B, digits=3))


===== MODEL B (Lifestyle + Cluster) — VALIDATION =====
Validation Accuracy: 0.9334
              precision    recall  f1-score   support

        High      0.958     0.957     0.957       806
         Low      0.950     0.953     0.952       443
      Medium      0.870     0.870     0.870       432

    accuracy                          0.933      1681
   macro avg      0.926     0.927     0.926      1681
weighted avg      0.933     0.933     0.933      1681


===== MODEL B (Lifestyle + Cluster) — TEST =====
Test Accuracy: 0.9437
              precision    recall  f1-score   support

        High      0.964     0.958     0.961       809
         Low      0.975     0.953     0.964       444
      Medium      0.877     0.908     0.892       433

    accuracy                          0.944      1686
   macro avg      0.939     0.939     0.939      1686
weighted avg      0.944     0.944     0.944      1686



In [9]:
# Cell 9 — Compare performance of Model A vs Model B (Test set)

print("===== FINAL TEST ACCURACY COMPARISON =====")
print(f"Model A (Lifestyle Only):      {test_acc_A:.4f}")
print(f"Model B (Lifestyle + Cluster): {test_acc_B:.4f}")

if test_acc_B > test_acc_A:
    print("\nAdding cluster labels improved stress-level classification.")
elif test_acc_B < test_acc_A:
    print("\nAdding cluster labels did not improve performance (cluster may be redundant).")
else:
    print("\nBoth models achieved the same test accuracy.")


===== FINAL TEST ACCURACY COMPARISON =====
Model A (Lifestyle Only):      0.9371
Model B (Lifestyle + Cluster): 0.9437

Adding cluster labels improved stress-level classification.


In [10]:
# Cell 10 — Save main model (Model B: lifestyle + cluster) and config for web app

import os
import joblib
import json

models_path = "../models"
os.makedirs(models_path, exist_ok=True)

# Save Model B (main)
joblib.dump(model_B, os.path.join(models_path, "multinomial_logreg_lifestyle_cluster.pkl"))

# Save feature config (for web app input order)
config = {
    "features_model_A": features_A,          # lifestyle only
    "features_model_B": features_B,          # lifestyle + cluster
    "target": target_col,
    "classes": sorted(list(y_train.unique()))   # e.g. ['High', 'Low', 'Medium'] depending on order
}

with open(os.path.join(models_path, "model_config.json"), "w") as f:
    json.dump(config, f, indent=4)

print("Model B and config saved to ../models")


Model B and config saved to ../models
